# Week 6 Advanced NLP: Named Entity Recognition, Topic Modeling & Transfer Learning

**BBC News Media Analytics Pipeline**
- **Repository Deliverable**: `notebooks/week6 advanced nlp.ipynb`
- **Dataset**: BBC News Articles (1,000 sample articles balanced across 5 news categories: Business, Entertainment, Politics, Sport, Technology)

---
## Notebook Structure
1. **Part 1 — Named Entity Recognition (NER)**: Extracting entities (`PERSON`, `ORG`, `GPE`/`LOC`, `DATE`, `NORP`) using spaCy (`en_core_web_sm`), entity distribution analysis across the entire corpus and news categories, and 5 formatted example articles with entity breakdowns.
2. **Part 2 — Topic Modeling**: Unsupervised discovery of latent themes across the corpus using **Latent Dirichlet Allocation (LDA)** ($k=5$), topic keyword visualization, cross-tabulation alignment with ground-truth news categories, and perplexity evaluation.
3. **Part 3 — Text Classification using Transfer Learning**: Contextual embedding classification with **DistilBERT (`distilbert-base-uncased`)** compared against a **Traditional Machine Learning Baseline (TF-IDF + Logistic Regression)**, complete evaluation, and saving the best model artifact.


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import joblib

import spacy
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix

import torch
from transformers import AutoTokenizer, AutoModel

# Plotting config
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11
print("All NLP and Machine Learning libraries imported successfully!")


All NLP and Machine Learning libraries imported successfully!


In [ ]:
# Load Dataset from data/bbc_news.csv
df = pd.read_csv('../data/bbc_news.csv')
print(f"Dataset Shape: {df.shape}")
print("\nCategory Counts:\n", df['category'].value_counts())
df.head()


Dataset Shape: (1000, 9)

Category Counts:
category
business         200
entertainment    200
politics         200
sport            200
technology       200


---
## Part 1 — Named Entity Recognition (NER)
In this section, we apply spaCy's `en_core_web_sm` model to extract entities (e.g. `PERSON`, `ORG`, `GPE` locations, `DATE`, `NORP`) across all articles. We analyze top entity types overall and breakdown entity distributions by news category.


In [ ]:
# Load spaCy NLP Engine
nlp = spacy.load('en_core_web_sm')

def extract_entities(text):
    doc = nlp(str(text)[:1000])
    return [{'text': ent.text, 'label': ent.label_} for ent in doc.ents]

df['entities'] = df['text'].apply(extract_entities)

all_ents = []
for idx, row in df.iterrows():
    for ent in row['entities']:
        all_ents.append({'category': row['category'], 'entity': ent['text'], 'label': ent['label']})

ent_df = pd.DataFrame(all_ents)
print(f"Total Entities Extracted across corpus: {len(ent_df)}")
print("\nTop Entity Types:\n", ent_df['label'].value_counts().head(8))


Total Entities Extracted across corpus: 3718

Top Entity Types:
label
PERSON      1108
ORG          752
GPE          679
DATE         346
CARDINAL     277
NORP         146
ORDINAL      112
EVENT         74


In [ ]:
# Visualize Top Entity Types & Breakdown by News Category
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top Entity Labels
label_counts = ent_df['label'].value_counts().head(8)
sns.barplot(x=label_counts.values, y=label_counts.index, ax=axes[0], palette="viridis", hue=label_counts.index, legend=False)
axes[0].set_title("Top 8 Named Entity Types in BBC Dataset", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Frequency Count")

# Entity Breakdown by Category
cat_label_df = ent_df.groupby(['category', 'label']).size().unstack(fill_value=0)
top_labels = label_counts.head(5).index
cat_label_df[top_labels].plot(kind='bar', stacked=True, ax=axes[1], colormap="Set2")
axes[1].set_title("Top Entity Type Breakdown by News Category", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Entity Count")
axes[1].set_xlabel("News Category")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Display 5 Example Articles with Extracted Entities Highlighted/Listed
sample_articles = df.groupby('category').first().reset_index()

for idx, row in sample_articles.iterrows():
    print(f"\n--- [ARTICLE {idx+1}] Category: {row['category'].upper()} | Title: '{row['title']}' ---")
    print(f"Description: {row['description']}")
    ents = row['entities'][:6]
    ent_str = ", ".join([f"{e['text']} ({e['label']})" for e in ents])
    print(f"Extracted Entities: {ent_str if ent_str else 'None'}")



--- [ARTICLE 1] Category: BUSINESS | Title: 'Scottish bakery Morton's Rolls 'ceases trading'' ---
Description: Companies House said it could strike off the bakery after it missed a deadline to file accounts.
Extracted Entities: Scottish (NORP), Morton (PERSON), Rolls (ORG), Companies House (ORG)

--- [ARTICLE 2] Category: ENTERTAINMENT | Title: 'TikTok trainspotter Francis Bourgeois: 'Passion is cool'' ---
Description: The star, fresh from a UK rail adventure for the BBC, talks identity, hate and saving the railway.
Extracted Entities: TikTok (ORG), Francis Bourgeois (PERSON), UK (GPE), BBC (ORG)

--- [ARTICLE 3] Category: POLITICS | Title: 'Two by-elections, two Labour wins... in two minutes' ---
Description: Keir Starmer's party overturned big Conservative majorities in Wellingborough and Kingswood.
Extracted Entities: Two (CARDINAL), two (CARDINAL), Labour (ORG), two minutes (TIME), Keir Starmer's (PERSON), Conservative (NORP)

--- [ARTICLE 4] Category: SPORT | Title: 'England v So

---
## Part 2 — Topic Modeling

### Topic Count Justification ($k=5$)
We selected **$k=5$ topics** for the unsupervised topic modeling task. This choice is specifically justified because the BBC news ground-truth corpus spans **five primary domain categories**: `business`, `entertainment`, `politics`, `sport`, and `technology`. Selecting $k=5$ allows direct mapping and quantitative alignment analysis using cross-tabulation.


In [ ]:
# Stopwords & Tokenization
stop_words = set(stopwords.words('english')).union({'said', 'mr', 'year', 'would', 'also', 'new', 'one', 'two', 'last', 'first', 'people', 'us', 'bbc', 'could', 'says'})

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    tokens = text.split()
    return ' '.join([t for t in tokens if len(t) > 2 and t not in stop_words])

df['clean_text'] = df['text'].apply(preprocess_text)

# Vectorization & LDA
vectorizer = CountVectorizer(max_df=0.95, min_df=3, max_features=1500)
tf_matrix = vectorizer.fit_transform(df['clean_text'])
tf_features = vectorizer.get_feature_names_out()

lda_model = LatentDirichletAllocation(n_components=5, max_iter=25, random_state=42, learning_method='online')
lda_output = lda_model.fit_transform(tf_matrix)

df['dominant_topic'] = [f"Topic {t+1}" for t in np.argmax(lda_output, axis=1)]

# Display Discovered Keywords
print("Discovered Keywords per Topic:")
for topic_idx, topic in enumerate(lda_model.components_):
    top_kw = [tf_features[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic {topic_idx+1}: {', '.join(top_kw)}")


Discovered Keywords per Topic:
Topic 1: twitter, former, england, elon, labour, musk, rules, tells, government, election
Topic 2: world, cup, best, years, data, games, star, time, back, gaming
Topic 3: win, league, manchester, england, city, day, open, united, final, championship
Topic 4: union, war, ban, tech, ukraine, strikes, week, set, say, facebook
Topic 5: tory, sunak, energy, rishi, prices, tax, pay, truss, say, cost


In [ ]:
# Evaluate Quality
perplexity_score = lda_model.perplexity(tf_matrix)
print(f"LDA Model Perplexity Score: {perplexity_score:.2f}")

# Cross Tabulation with Ground Truth News Categories
cross_tab_pct = pd.crosstab(df['category'], df['dominant_topic'], normalize='index') * 100

plt.figure(figsize=(9, 5))
sns.heatmap(cross_tab_pct, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': 'Percentage (%)'})
plt.title("Discovered LDA Topic vs Actual BBC News Category Alignment (%)", fontsize=13, fontweight='bold')
plt.xlabel("Discovered Topic")
plt.ylabel("Ground Truth Category")
plt.show()


LDA Model Perplexity Score: 1340.85


---
## Part 3 — Text Classification using Transfer Learning

### Strategy Justification (DistilBERT vs Traditional Baseline ML)
We selected **DistilBERT (`distilbert-base-uncased`)** as our Transfer Learning model. DistilBERT is a lightweight, distilled transformer pre-trained on generic text corpora that produces rich contextual embeddings. We extract the 768-dimensional mean pooling embeddings from the final hidden layer and train a Logistic Regression classifier head.

We benchmark Transfer Learning against a **Traditional Machine Learning Baseline (TF-IDF + Logistic Regression)** to compare Accuracy, Precision, Recall, F1-Score, and Confusion Matrix patterns.


In [ ]:
# Train / Test Split
X = df['text']
y = df['category']
label_names = sorted(y.unique())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Baseline ML Model (TF-IDF + Logistic Regression)
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_tr_tfidf = tfidf_vec.fit_transform(X_train)
X_te_tfidf = tfidf_vec.transform(X_test)

base_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
base_clf.fit(X_tr_tfidf, y_train)
y_pred_base = base_clf.predict(X_te_tfidf)

acc_base = accuracy_score(y_test, y_pred_base)
p_base, r_base, f1_base, _ = precision_recall_fscore_support(y_test, y_pred_base, average='weighted')

# 2. Transfer Learning Model (DistilBERT Embeddings + Logistic Regression)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
transformer = AutoModel.from_pretrained("distilbert-base-uncased")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transformer.to(device)
transformer.eval()

def embed_texts(texts_series, batch_size=64):
    all_embs = []
    text_list = texts_series.tolist()
    for i in range(0, len(text_list), batch_size):
        b_texts = text_list[i:i+batch_size]
        inp = tokenizer(b_texts, padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
        with torch.no_grad():
            out = transformer(**inp)
            emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embs.append(emb)
    return np.vstack(all_embs)

X_tr_emb = embed_texts(X_train)
X_te_emb = embed_texts(X_test)

tl_clf = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
tl_clf.fit(X_tr_emb, y_train)
y_pred_tl = tl_clf.predict(X_te_emb)

acc_tl = accuracy_score(y_test, y_pred_tl)
p_tl, r_tl, f1_tl, _ = precision_recall_fscore_support(y_test, y_pred_tl, average='weighted')

print(f"Baseline TF-IDF      -> Accuracy: {acc_base:.4f}, F1-Score: {f1_base:.4f}")
print(f"Transfer Learning    -> Accuracy: {acc_tl:.4f}, F1-Score: {f1_tl:.4f}")


Baseline TF-IDF      -> Accuracy: 0.7400, F1-Score: 0.7345
Transfer Learning    -> Accuracy: 0.8450, F1-Score: 0.8447


In [ ]:
# Visualization Comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

comp_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'] * 2,
    'Score': [acc_base, p_base, r_base, f1_base, acc_tl, p_tl, r_tl, f1_tl],
    'Approach': ['Baseline (TF-IDF)'] * 4 + ['Transfer Learning (DistilBERT)'] * 4
})
sns.barplot(data=comp_df, x='Metric', y='Score', hue='Approach', ax=axes[0], palette="Set1")
axes[0].set_ylim(0.7, 1.0)
axes[0].set_title("Performance Comparison: Baseline vs Transfer Learning", fontsize=13, fontweight='bold')
for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2., h), ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

cm = confusion_matrix(y_test, y_pred_tl, labels=label_names)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title("Transfer Learning (DistilBERT) Confusion Matrix", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Predicted Category")
axes[1].set_ylabel("True Category")

plt.tight_layout()
plt.show()


In [ ]:
# Save Best Performing Model
best_model_artifact = {
    'model': tl_clf if f1_tl >= f1_base else base_clf,
    'tokenizer_name': 'distilbert-base-uncased',
    'approach': 'Transfer Learning (DistilBERT + Logistic Head)' if f1_tl >= f1_base else 'Baseline (TF-IDF + Logistic Head)',
    'categories': label_names,
    'metrics': {
        'baseline_accuracy': acc_base, 'baseline_f1': f1_base,
        'transfer_learning_accuracy': acc_tl, 'transfer_learning_f1': f1_tl
    }
}

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model_artifact, '../models/best_nlp_model.joblib')
print("Successfully saved best performing model artifact to ../models/best_nlp_model.joblib!")


Successfully saved best performing model artifact to ../models/best_nlp_model.joblib!
